# Phase 3 — RQ1 prior comparison (no fine-tuning)

This notebook compares prior matrices with the B0 forgetting trajectories already recorded in the repository. It does **not** fine-tune a model. Enable Kaggle Internet to clone the repository, download XNLI/model files for the embedding prior, and retrieve URIEL+/Grambank source data. The embedding encoder can run on CPU; a GPU only speeds up inference.

**Required:** attach a Kaggle input dataset containing the numeric `interference_matrix.csv`. The published paper defines rows as affected/past languages and columns as added/incoming languages; its negative values are converted to risk using `max(0, -score)`. The paper page currently exposes a figure, not a downloadable numeric matrix, so this notebook does not invent or digitize matrix values. The notebook generates URIEL+ and embedding matrices if they are not attached. It checks Grambank coverage and will not make a matrix if the official release lacks a pilot language.

For this pilot, Grambank is intentionally excluded because v1.0.3 lacks Spanish and German; the coverage report and exclusion reason are recorded. No values are imputed or substituted. Other missing inputs are marked in the analysis summary.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
import json
import shutil
import subprocess
import sys
import zipfile

REPO_URL = "https://github.com/Mani-S-49/asymmetric-interference-continual-learning..git"
REPO_ROOT = Path("/kaggle/working/asymmetric-interference-continual-learning")
if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)
subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)
REPO_REVISION = subprocess.check_output(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True
).strip()
sys.path.insert(0, str(REPO_ROOT))

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/phase3_rq1_output")
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True)

def find_input(filename):
    matches = sorted(INPUT_ROOT.rglob(filename))
    if len(matches) > 1:
        raise ValueError(f"Found multiple {filename} files: {matches}")
    return matches[0] if matches else None

prior_files = {
    "interference_matrix": find_input("interference_matrix.csv"),
    "uriel_distance": find_input("uriel_syntactic_distance.csv"),
    "grambank_distance": None,
    "embedding_similarity": find_input("embedding_similarity.csv"),
}
if prior_files["interference_matrix"] is None:
    raise FileNotFoundError(
        "Attach a Kaggle input containing the user-verified "
        "interference_matrix.csv before running this notebook."
    )

def install_if_missing(module_name, package_spec):
    if importlib.util.find_spec(module_name) is None:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", package_spec],
            check=True,
        )

if prior_files["uriel_distance"] is None:
    install_if_missing("urielplus", "urielplus==1.3.3")
    generated_uriel = OUTPUT_DIR / "uriel_syntactic_distance.csv"
    subprocess.run(
        [sys.executable, "-m", "phase3.prepare_urielplus_distance",
         "--output", str(generated_uriel)],
        cwd=REPO_ROOT,
        check=True,
    )
    prior_files["uriel_distance"] = generated_uriel

attached_grambank = find_input("grambank_distance.csv")
if attached_grambank is not None:
    print("Ignoring attached Grambank matrix: it is excluded by the current pilot design.")
if prior_files["grambank_distance"] is None:
    grambank_dir = OUTPUT_DIR / "grambank"
    result = subprocess.run(
        [sys.executable, "-m", "phase3.prepare_grambank_distance",
         "--output-dir", str(grambank_dir)],
        cwd=REPO_ROOT,
        text=True,
        capture_output=True,
    )
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print("Grambank unavailable for this pilot:", result.stderr[-2000:])
    generated_grambank = grambank_dir / "grambank_distance.csv"
    if generated_grambank.is_file():
        prior_files["grambank_distance"] = generated_grambank
    elif (grambank_dir / "grambank_coverage.json").is_file():
        shutil.copy2(
            grambank_dir / "grambank_coverage.json",
            OUTPUT_DIR / "grambank_coverage.json",
        )

if prior_files["embedding_similarity"] is None:
    install_if_missing("datasets", "datasets")
    install_if_missing("sentence_transformers", "sentence-transformers")
    generated_embedding = OUTPUT_DIR / "embedding_similarity.csv"
    subprocess.run(
        [sys.executable, "-m", "phase3.prepare_embedding_similarity",
         "--protocol", str(REPO_ROOT / "phase2" / "configs" / "pilot_protocol.json"),
         "--prepared-data-dir", str(OUTPUT_DIR / "embedding_xnli_data"),
         "--output", str(generated_embedding)],
        cwd=REPO_ROOT,
        check=True,
    )
    prior_files["embedding_similarity"] = generated_embedding

print({name: str(path) if path else "MISSING" for name, path in prior_files.items()})
print("Repository revision:", REPO_REVISION)

## Run the analysis

The B0 result JSONs are committed under `phase2/runs/b0_kaggle_2026-11-11/results`. Script and broad-family mismatch priors are derived from the pilot language metadata. URIEL+ and embedding inputs are generated when not attached; Grambank is included only if the selected release covers all five languages.

In [ ]:
command = [
    sys.executable, "-m", "phase3.analyze_priors",
    "--results-dir", str(REPO_ROOT / "phase2" / "runs" / "b0_kaggle_2026-11-11" / "results"),
    "--output-dir", str(OUTPUT_DIR),
    "--exclude-prior", "grambank_distance",
]
for key, option in (
    ("interference_matrix", "--interference-matrix"),
    ("uriel_distance", "--uriel-distance"),
    ("embedding_similarity", "--embedding-similarity"),
):
    if prior_files[key] is not None:
        command.extend([option, str(prior_files[key])])
print("Running:", " ".join(command))
subprocess.run(command, cwd=REPO_ROOT, check=True)

summary = json.loads((OUTPUT_DIR / "summary.json").read_text(encoding="utf-8"))
print(json.dumps(summary, indent=2))

## Package results

The ZIP contains analysis tables, run metadata, generated prior matrices, and the Grambank coverage report. It does not duplicate attached input matrices. Send the resulting ZIP back for interpretation. If any prior appears under `missing_prior_inputs`, attach or generate that input and rerun.

In [ ]:
input_hashes = {}
for name, path in prior_files.items():
    if path is not None:
        input_hashes[name] = hashlib.sha256(path.read_bytes()).hexdigest()
run_metadata = {
    "analysis": "phase3_rq1_prior_comparison",
    "repository_revision": REPO_REVISION,
    "gpu_required": False,
    "training_performed": False,
    "prior_input_sha256": input_hashes,
    "missing_prior_inputs": summary["missing_prior_inputs"],
}
(OUTPUT_DIR / "run_metadata.json").write_text(
    json.dumps(run_metadata, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)
ZIP_PATH = Path("/kaggle/working/phase3_rq1_results.zip")
if ZIP_PATH.exists():
    ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for file_path in sorted(OUTPUT_DIR.iterdir()):
        if file_path.is_file():
            archive.write(file_path, arcname=file_path.name)
print("Result archive:", ZIP_PATH)
print("Download it from the Kaggle notebook output files.")